In [26]:
from datasets import load_dataset
from tqdm import tqdm

filename = "./base2/english_wikipedia.txt"

ds = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-raw-v1",
    split="train"
)

for item in ds:
    print(item["text"][:500])
    break

with open(filename, "w", encoding="utf-8") as f:
    for i, item in enumerate(tqdm(ds)):
        if i >= 350000:
            break
        f.write(item["text"] + "\n")


 19%|█▉        | 350000/1801350 [00:07<00:29, 49328.98it/s]


In [27]:
from os import walk
import re
from tqdm import tqdm

base_path = './base2/'

data_text = ''

for dirpath, dirnames, filenames in walk(base_path):
    for file_name in filenames:
        if file_name[-4:] != '.txt': continue

        print('Read:', file_name)
        try:
            current_text = open(dirpath + '/' + file_name, 'r', encoding='utf-8').read()
            data_text += current_text
        except Exception as e:
            print(f'Error reading {file_name}: {e}')

import pymorphy3

morph = pymorphy3.MorphAnalyzer()

_lemma_cache = {}

def prepare_text(text):
    n_text = text.lower()
    n_text = re.sub(r'[^a-z\s]+', ' ', n_text)
    n_text = re.sub(r'\n+', ' ', n_text)
    n_text = re.sub(r'\s+', ' ', n_text)

    return n_text

data_text = prepare_text(data_text)

alphabet = list(set(data_text))

print('---')
print('Final data length:', len(data_text), 'symbols')
print('Alphabet length:', len(alphabet))
print('Alphabet:', alphabet)
print('Text sample:', data_text[:100])

Read: english_wikipedia.txt
---
Final data length: 94700738 symbols
Alphabet length: 27
Alphabet: ['m', 'i', 'b', 'w', 'o', 'q', 'c', 'r', 'u', 'f', 's', 'g', 'v', 'h', 'n', ' ', 'a', 't', 'k', 'y', 'e', 'p', 'j', 'l', 'z', 'x', 'd']
Text sample:  valkyria chronicles iii senj no valkyria unrecorded chronicles japanese lit valkyria of the battlef


In [56]:
from collections import defaultdict, Counter
from typing import List, Dict, Set

words = data_text.split()
freq = Counter(words)

min_frequency = 10

CLEAR_MEMORY = True

vocab = {w for w, c in freq.items() if c >= min_frequency}

print('words in data:', len(words))
print(f"Before word count: {len(freq)}")
print(f"Vocabulary size: {len(vocab)}")

words in data: 16133765
Before word count: 198888
Vocabulary size: 46776


In [57]:
from tqdm import tqdm
import gc

word_to_id = {w: i for i, w in enumerate(sorted(vocab))}
# id_to_word = {i: w for w, i in word_to_id.items()}

context_to_words: Dict[int, Set[int]] = defaultdict(set)
# context_to_context: Dict[int, Set[int]] = defaultdict(set)

radius = 3

for i, w in enumerate(tqdm(words)):
    if i < radius or i >= len(words) - radius:
        continue
    if w not in vocab:
        continue

    w_id = word_to_id[w]
    left = max(0, i - radius)
    right = min(len(words), i + radius + 1)

    context_word_ids = set()
    for j in range(left, right):
        if j == i:
            continue
        cand = words[j]
        if cand in vocab:
            context_word_ids.add(word_to_id[cand])

    if not context_word_ids:
        continue

    ctx_key = frozenset(context_word_ids)
    ctx_id = hash(ctx_key)

    context_to_words[ctx_id].add(w_id)
    # context_to_context[ctx_id] = context_word_ids

print(f"Total contexts: {len(context_to_words)}")

if CLEAR_MEMORY:
    del words
    gc.collect()

100%|██████████| 16133765/16133765 [01:37<00:00, 165523.86it/s]


Total contexts: 15363701


In [58]:
import matplotlib.pyplot as plt

# количество контекстов для каждого слова
word_context_counts = dict()
for ctx_id, word_ids in context_to_words.items():
    for w_id in word_ids:
        word_context_counts[w_id] = word_context_counts.get(w_id, 0) + 1

word_context_counts = sorted(word_context_counts.items(), key=lambda x: x[1], reverse=True)

mean_contexts = sum(count for word, count in word_context_counts) / len(word_context_counts)
print(f"Mean number of contexts per word: {mean_contexts:.2f}")

std_contexts = (sum((count - mean_contexts) ** 2 for word, count in word_context_counts) / len(word_context_counts)) ** 0.5
print(f"Standard deviation of contexts per word: {std_contexts:.2f}")

print(f"{mean_contexts:.2f} ± {std_contexts:.2f}")

# plt.plot([count for word, count in word_context_counts if count > 100])
# plt.xlabel('Number of Contexts')
# plt.ylabel('Frequency')
# plt.title('Distribution of Word-Context Counts')
# plt.show()

if CLEAR_MEMORY:
    del word_context_counts
    gc.collect()

Mean number of contexts per word: 329.73
Standard deviation of contexts per word: 7544.83
329.73 ± 7544.83


In [59]:
# количество слов для каждого контекста

context_word_counts = {ctx_id: len(word_ids) for ctx_id, word_ids in context_to_words.items()}

context_word_counts = sorted(context_word_counts.items(), key=lambda x: x[1], reverse=True)

mean_words = sum(count for ctx, count in context_word_counts) / len(context_word_counts)
print(f"Mean number of words per context: {mean_words:.2f}")

std_words = (sum((count - mean_words) ** 2 for ctx, count in context_word_counts) / len(context_word_counts)) ** 0.5
print(f"Standard deviation of words per context: {std_words:.2f}")

print(f"{mean_words:.2f} ± {std_words:.2f}")

# plt.plot([count for ctx, count in context_word_counts if count > 1])
# plt.xlabel('Number of Words')
# plt.ylabel('Frequency')
# plt.title('Distribution of Context-Word Counts')
# plt.show()

if CLEAR_MEMORY:
    del context_word_counts
    gc.collect()

Mean number of words per context: 1.00
Standard deviation of words per context: 0.10
1.00 ± 0.10


In [60]:
blocked: Dict[int, Set[int]] = defaultdict(set)

for ctx_id, ctx_words in tqdm(context_to_words.items()):
    ctx_words_list = list(ctx_words)
    for a in ctx_words_list:
        for b in ctx_words_list:
            if a != b:
                blocked[a].add(b)

if CLEAR_MEMORY:
    del context_to_words
    gc.collect()

class_members: List[Set[int]] = []
word_to_class: Dict[str, int] = {}

sorted_words = sorted(vocab, key=lambda x: (freq[x], x))

for w in tqdm(sorted_words):
    w_id = word_to_id[w]

    placed = False
    for class_id, members in enumerate(class_members):
        if not (blocked[w_id] & members):
            members.add(w_id)
            word_to_class[w] = class_id
            placed = True
            break

    if not placed:
        class_members.append({w_id})
        word_to_class[w] = len(class_members) - 1

classes: Dict[int, List[str]] = {}
for w, cls in word_to_class.items():
    classes.setdefault(cls, []).append(w)

for cls in classes:
    classes[cls].sort()

100%|██████████| 46776/46776 [00:00<00:00, 336131.27it/s]


In [61]:
print('classes:', len(classes))

print(', '.join(str(len(words)) for words in classes.values()))

for cls_id, words_in_class in classes.items():
    print(f"Class {cls_id}: {len(words_in_class)} words")
    print(words_in_class[:20]) 
    print('---')

classes: 111
40440, 3171, 998, 536, 321, 211, 145, 111, 95, 72, 61, 46, 39, 33, 29, 27, 25, 20, 17, 17, 14, 12, 11, 11, 11, 10, 8, 6, 7, 7, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6, 5, 6, 5, 6, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 3, 3, 2, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 2, 3, 3, 2, 2, 3, 3, 2, 2, 2, 2, 1, 3, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1
Class 0: 40440 words
['aaa', 'aaaaa', 'aachen', 'aadt', 'aalborg', 'aaliyah', 'aamir', 'aang', 'aarhus', 'aaron', 'aasho', 'aashto', 'aat', 'aau', 'aayirathil', 'ab', 'aba', 'ababa', 'aback', 'abadeer']
---
Class 1: 3171 words
['aa', 'abandonment', 'abdallah', 'aberdeen', 'abolition', 'absolute', 'abu', 'abundant', 'abydos', 'ac', 'accept', 'accessed', 'accessible', 'accident', 'accomplishments', 'accordingly', 'accounting', 'accounts', 'acetyl', 'achievement']
---
Class 2: 998 words
['aafc', 'abandoned', 'abbey', 'abbreviated', 'abby', 'abduction', 'acacia', 'accompany', 'accused', 'acting', 'active', 'actors'

In [62]:
def encode_sentence_to_classes(sentence: str, word_to_class: dict) -> list[int | None]:
    tokens = sentence.lower().split()
    result = []

    for token in tokens:
        cls_id = word_to_class.get(token)
        result.append(cls_id)

    return result

sample_sentence = "результат сжатия текста с использованием классов омонимов"
encoded = encode_sentence_to_classes(sample_sentence, word_to_class)

print("Sentence:", sample_sentence)
print("Encoded:", encoded)

sample_sentence = "замок лук"
encoded = encode_sentence_to_classes(sample_sentence, word_to_class)

print("Sentence:", sample_sentence)
print("Encoded:", encoded)

Sentence: результат сжатия текста с использованием классов омонимов
Encoded: [None, None, None, None, None, None, None]
Sentence: замок лук
Encoded: [None, None]
